<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-dark.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/ws-clustering.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Welcome to the WeightsLab <b>Face Recognition (triplet loss)</b> notebook! <a href="https://github.com/GrayboxTech/weightslab">WeightsLab</a> traces training signals back to the exact samples producing them. Browse the <a href="https://grayboxtech.github.io/weightslab/latest/index.html">Docs</a> for details.</div>

# Face Recognition (triplet loss) with WeightsLab

Trains a ResNet-18 embedding head with online batch-hard **triplet loss** and tracks verification / retrieval / similarity signals per sample.

> Data: Defaults to the **Olivetti Faces** dataset (via scikit-learn) which works **offline** - no download needed. Switch to LFW or a folder dataset in the config.

This notebook is fully **self-contained**: the `face/` helper modules and the training loop
are inlined below, so there is no repo to clone and no `main.py` to shell out to. Run the
cells top to bottom.

## Setup

On Colab, pick a **T4 GPU** runtime (`Runtime -> Change runtime type -> T4 GPU`).

<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?color=5865F2&logo=pypi&logoColor=white" alt="PyPI - Version"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/dm/weightslab?color=5865F2" alt="PyPI - Downloads"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/pyversions/weightslab?color=5865F2&logo=python&logoColor=white" alt="PyPI - Python Version"></a>

In [ ]:
# Install WeightsLab. Colab already ships torch, torchvision, numpy, scikit-learn
# and Pillow, so nothing extra is needed here.
# %pip install -q weightslab
%pip install --pre --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "weightslab==1.3.3.dev3"

## 1. Imports

`weightslab` is imported as `wl`. The two `guard_*_context` managers scope a block as
training vs. evaluation so signals are attributed to the right phase. We also detect the
compute `device`.

In [ ]:
import logging
import os
import tempfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torchvision import models
import torchvision.transforms as T

from typing import Any, Dict, List, Optional, Tuple

import weightslab as wl
from weightslab.components.global_monitoring import (
    guard_training_context,
    guard_testing_context,
)

logging.basicConfig(level=logging.ERROR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Face helper modules (inlined)

These are the `face/` package modules from the original example, inlined here so the
notebook is self-contained. Only the intra-package `from face. ...` imports were removed -
every class and function is otherwise unchanged. Order follows the dependencies:
**utils -> data -> model -> signals**.

In [ ]:
# ===== face/utils.py =====
"""
Utility functions for face recognition training.
- Pairwise distance computation
- Online batch-hard triplet mining
- Verification and Rank-1 evaluation helpers
- Similarity grouping signals for clustering-oriented analysis
"""

import numpy as np
import torch

from typing import Tuple, Dict, List, Any


# ============================================================
# Distance helpers
# ============================================================

def pairwise_distances(embeddings: torch.Tensor, squared: bool = False) -> torch.Tensor:
    """Compute pairwise L2 distance matrix for a batch of embeddings.

    Args:
        embeddings: (B, D) tensor
        squared: return squared L2 distances when True

    Returns:
        (B, B) distance matrix
    """
    dot = torch.matmul(embeddings, embeddings.t())
    sq_norms = torch.diagonal(dot) # (B,)
    distances = sq_norms.unsqueeze(0) - 2.0 * dot + sq_norms.unsqueeze(1)
    distances = distances.clamp(min=0.0)

    if not squared:
        # Avoid NaN gradients at exactly 0
        mask = (distances == 0.0).float()
        distances = distances + mask * 1e-16
        distances = torch.sqrt(distances)
        distances = distances * (1.0 - mask)

    return distances


# ============================================================
# Triplet mining
# ============================================================

def mine_batch_hard(
    embeddings: torch.Tensor,
    labels: torch.Tensor,
    squared: bool = False,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Online batch-hard triplet mining (Hermans et al., 2017).

    For each anchor selects:
    - Hardest positive : same-class sample with the **largest** distance
    - Hardest negative : different-class sample with the **smallest** distance

    Args:
        embeddings: (B, D) detached from graph during mining
        labels: (B,) integer class ids
        squared: use squared L2 distances for mining

    Returns:
        anc_idx, pos_idx, neg_idx 1-D LongTensors; only valid anchors
        (those that have at least one positive peer) are included.
    """
    dist_mat = pairwise_distances(embeddings.detach(), squared=squared)
    B = labels.shape[0]
    device = labels.device

    same = labels.unsqueeze(0) == labels.unsqueeze(1) # (B, B)
    diff = ~same
    eye = torch.eye(B, dtype=torch.bool, device=device)

    # Hardest positive (exclude self)
    pos_mask = same & ~eye
    pos_dist = dist_mat * pos_mask.float()
    _, pos_idx = pos_dist.max(dim=1)

    # Hardest negative (mask invalid positions with large value)
    neg_dist = dist_mat + (~diff).float() * 1e9
    _, neg_idx = neg_dist.min(dim=1)

    # Keep only anchors that have at least one positive peer in the batch
    valid = pos_mask.any(dim=1)
    anc_idx = torch.where(valid)[0]

    return anc_idx, pos_idx[anc_idx], neg_idx[anc_idx]


# ============================================================
# Numpy-level evaluation helpers
# ============================================================

def compute_verification_metrics(
    embeddings: np.ndarray,
    labels: np.ndarray,
    thresholds: np.ndarray | None = None,
) -> dict:
    """Pairwise verification accuracy over a threshold grid."""
    if thresholds is None:
        thresholds = np.linspace(0.0, 2.0, 100)

    n = len(embeddings)

    # Pairwise L2 distances
    dot = embeddings @ embeddings.T # (N, N)
    sq = np.sum(embeddings ** 2, axis=1)
    dist_mat = (sq[:, None] - 2.0 * dot + sq[None, :]).clip(min=0.0)
    dist_mat = np.sqrt(dist_mat.clip(min=1e-16)) * (dist_mat != 0.0)

    same_pair = labels[:, None] == labels[None, :] # (N, N)

    # Upper triangle only (avoid double-counting)
    iu = np.triu_indices(n, k=1)
    dist_pairs = dist_mat[iu]
    is_same_pair = same_pair[iu]

    n_same = int(is_same_pair.sum())
    n_diff = int((~is_same_pair).sum())

    best_acc = 0.0
    best_threshold = float(thresholds[len(thresholds) // 2])
    best_far = 0.0
    best_frr = 0.0

    for thr in thresholds:
        pred_same = dist_pairs <= thr
        tp = int((pred_same & is_same_pair).sum())
        fp = int((pred_same & ~is_same_pair).sum())
        fn = int((~pred_same & is_same_pair).sum())
        tn = int((~pred_same & ~is_same_pair).sum())

        total = n_same + n_diff
        acc = (tp + tn) / total if total > 0 else 0.0
        far = fp / n_diff if n_diff > 0 else 0.0
        frr = fn / n_same if n_same > 0 else 0.0

        if acc > best_acc:
            best_acc = acc
            best_threshold = float(thr)
            best_far = far
            best_frr = frr

    return {
        "verification_accuracy": float(best_acc),
        "best_threshold": float(best_threshold),
        "far": float(best_far),
        "frr": float(best_frr),
    }


def compute_rank1_accuracy(embeddings: np.ndarray, labels: np.ndarray) -> float:
    """1-NN Rank-1 retrieval accuracy (leave-one-out)."""
    n = len(embeddings)
    dot = embeddings @ embeddings.T
    sq = np.sum(embeddings ** 2, axis=1)
    dist_mat = (sq[:, None] - 2.0 * dot + sq[None, :]).clip(min=0.0)

    # Exclude self
    np.fill_diagonal(dist_mat, 1e9)
    nn_idx = dist_mat.argmin(axis=1)

    correct = int(np.sum(labels[nn_idx] == labels))
    return correct / n


def compute_similarity_grouping(
    embeddings: np.ndarray,
    uids: List[str],
    cluster_eps: float = 0.6,
    cluster_min_samples: int = 2,
) -> Dict[str, Any]:
    """Build per-sample similarity grouping signals for test-set clustering.

    Returns per-sample arrays that can be written as WeightsLAB signals:
    - cluster_id: DBSCAN cluster index (-1 for noise)
    - cluster_size: size of assigned cluster (1 for noise)
    - nn1_uid: nearest-neighbor UID in the evaluated set
    - nn1_distance: nearest-neighbor L2 distance
    """
    n = len(embeddings)
    if n == 0:
        return {
            "cluster_id": np.array([], dtype=np.int32),
            "cluster_size": np.array([], dtype=np.int32),
            "nn1_uid": [],
            "nn1_distance": np.array([], dtype=np.float32),
            "num_clusters": 0.0,
            "noise_ratio": 0.0,
            "mean_nn1_distance": float("nan"),
        }

    # Pairwise distances for NN lookup
    dot = embeddings @ embeddings.T
    sq = np.sum(embeddings ** 2, axis=1)
    dist_mat = (sq[:, None] - 2.0 * dot + sq[None, :]).clip(min=0.0)
    dist_mat = np.sqrt(dist_mat.clip(min=1e-16))

    if n == 1:
        nn_idx = np.array([0], dtype=np.int64)
        nn_dist = np.array([0.0], dtype=np.float32)
    else:
        np.fill_diagonal(dist_mat, 1e9)
        nn_idx = dist_mat.argmin(axis=1)
        nn_dist = dist_mat[np.arange(n), nn_idx].astype(np.float32)

    nn_uid = [uids[int(i)] for i in nn_idx.tolist()]

    # Unsupervised clustering on embeddings
    try:
        from sklearn.cluster import DBSCAN

        labels = DBSCAN(eps=cluster_eps, min_samples=cluster_min_samples, metric="euclidean").fit_predict(embeddings)
    except Exception:
        labels = np.full((n,), -1, dtype=np.int32)

    labels = labels.astype(np.int32)
    valid_labels = labels[labels >= 0]
    num_clusters = int(len(np.unique(valid_labels))) if valid_labels.size > 0 else 0

    counts = {}
    if valid_labels.size > 0:
        uniq, cnt = np.unique(valid_labels, return_counts=True)
        counts = {int(k): int(v) for k, v in zip(uniq.tolist(), cnt.tolist())}

    cluster_size = np.array([counts.get(int(c), 1) if int(c) >= 0 else 1 for c in labels], dtype=np.int32)
    noise_ratio = float(np.mean(labels < 0))

    return {
        "cluster_id": labels,
        "cluster_size": cluster_size,
        "nn1_uid": nn_uid,
        "nn1_distance": nn_dist,
        "num_clusters": float(num_clusters),
        "noise_ratio": noise_ratio,
        "mean_nn1_distance": float(nn_dist.mean()) if len(nn_dist) > 0 else float("nan"),
    }

In [ ]:
# ===== face/data.py =====
"""
FaceDataset — toy face recognition dataset.

Supported back-ends
-------------------
"olivetti" Olivetti Faces from sklearn (40 identities, 400 images, 64×64 grey).
            Self-contained; requires only scikit-learn. Good default toy set.
"lfw" Labeled Faces in the Wild (LFW) via torchvision. Downloaded on
            first use to *root*. Much larger but realistic.
"folder" Generic ImageFolder layout: root/{split}/{class_name}/*.jpg

Every sample is returned as:
    (image_tensor: Tensor[C,H,W],
     uid: str,
     label: int,
     metadata: dict)

These map directly onto the (data, uid, target, metadata) convention used
throughout the WeightsLAB kitchen examples so the same training loop works
without modification.
"""

import os
import logging

import numpy as np
import torch
from torch.utils.data import Dataset
from typing import Dict, Optional, Tuple

import torchvision.transforms as T

logger = logging.getLogger(__name__)


# ============================================================
# Dataset
# ============================================================

class FaceDataset(Dataset):
    """Unified face recognition dataset wrapper.

    Args:
        root: Download / data root (only used for lfw / folder).
        dataset_type: One of "olivetti", "lfw", "folder".
        split: "train" or "test" (ignored for pre-split sources).
        image_size: Spatial size; images are resized to (image_size, image_size).
        train_ratio: Fraction of per-class samples used for training
                               (Olivetti only).
        min_images_per_class: Classes with fewer samples are discarded.
        transform: Optional torchvision transform; defaults to
                               Resize → ToTensor → Normalize([0.5], [0.5]).
        seed: RNG seed for reproducible train/test splits.
    """

    def __init__(
        self,
        root: str = ".",
        dataset_type: str = "olivetti",
        split: str = "train",
        image_size: int = 64,
        train_ratio: float = 0.8,
        min_images_per_class: int = 2,
        transform=None,
        seed: int = 42,
    ):
        self.dataset_type = dataset_type
        self.split = split
        self.image_size = image_size
        self.transform = transform or self._default_transform(image_size)

        # These are populated by each loader
        self.images: Optional[np.ndarray] = None # (N, H, W) float [0,1] — Olivetti only
        self.img_paths: Optional[np.ndarray] = None
        self.labels: np.ndarray = np.array([], dtype=np.int64)
        self.num_classes: int = 0

        if dataset_type == "olivetti":
            self._load_olivetti(train_ratio, min_images_per_class, seed)
        elif dataset_type == "lfw":
            self._load_lfw(root, min_images_per_class, split)
        elif dataset_type == "folder":
            self._load_folder(root, split)
        else:
            raise ValueError(
                f"Unknown dataset_type {dataset_type!r}. "
                "Choose from 'olivetti', 'lfw', 'folder'."
            )

        logger.info(
            f"FaceDataset [{dataset_type} / {split}] "
            f"| samples={len(self)} | classes={self.num_classes}"
        )

    # ----------------------------------------------------------
    # Loaders
    # ----------------------------------------------------------

    @staticmethod
    def _default_transform(image_size: int):
        return T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

    def _load_olivetti(self, train_ratio: float, min_images: int, seed: int):
        """Load and split the Olivetti Faces dataset (sklearn)."""
        from sklearn.datasets import fetch_olivetti_faces

        data = fetch_olivetti_faces(shuffle=True, random_state=seed)
        images = data.images # (400, 64, 64) float [0,1]
        labels = data.target.astype(np.int64)

        # Drop classes with insufficient samples
        unique, counts = np.unique(labels, return_counts=True)
        valid_classes = unique[counts >= min_images]
        mask = np.isin(labels, valid_classes)
        images, labels = images[mask], labels[mask]

        # Remap labels to a contiguous 0…N-1 range
        mapping = {int(c): i for i, c in enumerate(sorted(valid_classes.tolist()))}
        labels = np.array([mapping[int(l)] for l in labels], dtype=np.int64)

        # Per-class stratified train/test split
        rng = np.random.RandomState(seed)
        train_idx, test_idx = [], []
        for cls in np.unique(labels):
            idx = np.where(labels == cls)[0]
            n_train = max(1, int(len(idx) * train_ratio))
            perm = rng.permutation(len(idx))
            train_idx.extend(idx[perm[:n_train]].tolist())
            test_idx.extend(idx[perm[n_train:]].tolist())

        indices = train_idx if self.split == "train" else test_idx
        self.images = images[indices]
        self.labels = labels[indices]
        self.num_classes = len(mapping)

    def _load_lfw(self, root: str, min_images: int, split: str):
        """Load LFW People via torchvision (downloads on first call)."""
        from torchvision.datasets import LFWPeople

        split_map = {"train": "train", "test": "test", "val": "10fold"}
        lfw_split = split_map.get(split, "train")
        ds = LFWPeople(root=root, split=lfw_split, download=True, transform=None)

        paths, lbls = zip(*ds.imgs)
        lbls = np.array(lbls, dtype=np.int64)

        # Filter low-shot identities
        unique, counts = np.unique(lbls, return_counts=True)
        valid = set(unique[counts >= min_images].tolist())
        mask = np.array([int(l) in valid for l in lbls])

        self.img_paths = np.array(paths)[mask]
        lbls = lbls[mask]

        mapping = {int(c): i for i, c in enumerate(sorted(valid))}
        self.labels = np.array([mapping[int(l)] for l in lbls], dtype=np.int64)
        self.num_classes = len(mapping)

    def _load_folder(self, root: str, split: str):
        """Load from a torchvision ImageFolder directory."""
        from torchvision.datasets import ImageFolder

        split_dir = os.path.join(root, split)
        ds = ImageFolder(split_dir)
        paths, lbls = zip(*ds.imgs)
        self.img_paths = list(paths)
        self.labels = np.array(lbls, dtype=np.int64)
        self.num_classes = len(ds.classes)

    # ----------------------------------------------------------
    # Dataset interface
    # ----------------------------------------------------------

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, str, int, Dict]:
        """Return (image_tensor, uid, label_id, metadata).

        The (data, uid, target, metadata) signature mirrors the WeightsLAB
        kitchen convention used in the VLA training example.
        """
        label = int(self.labels[idx])

        if self.dataset_type == "olivetti":
            from PIL import Image as PILImage
            img_np = self.images[idx] # (H, W) float [0,1]
            img_pil = PILImage.fromarray(
                (img_np * 255).astype(np.uint8), mode="L"
            ).convert("RGB")
        else:
            from PIL import Image as PILImage
            img_pil = PILImage.open(self.img_paths[idx]).convert("RGB")

        image_tensor = self.transform(img_pil)

        uid = f"{self.split}_cls{label:04d}_idx{idx:06d}"
        metadata = {
            "split": self.split,
            "label_id": label,
            "idx": idx,
            "dataset_type": self.dataset_type,
        }
        return image_tensor, uid, label, metadata

In [ ]:
# ===== face/model.py =====
"""
Face embedding model.

Architecture
------------
Pretrained backbone (ResNet-18 / ResNet-50 / MobileNet-V3-Small)
    ?
EmbeddingHead Linear ? BN ? ReLU ? Linear
    ?
L2-normalised D-dimensional embedding

The backbone is optionally frozen so that only the lightweight head is trained
(recommended toy-example setup). The combined graph is registered with
WeightsLAB for model tracking.

Public interface
----------------
FaceEmbeddingModel.get_embeddings(images) ? normalised embeddings (B, D)
FaceEmbeddingModel.train_step(images, labels, ...) ? scalar loss float
"""

import logging

import torch
import torch.nn as nn
import torch.nn.functional as F

import weightslab as wl

from torchvision import models
from typing import List


logger = logging.getLogger(__name__)


# ============================================================
# Network modules
# ============================================================

class EmbeddingHead(nn.Module):
    """Projection head: backbone_feature_dim ? embedding_dim (L2-normalised)."""

    def __init__(self, in_dim: int, hidden_dim: int, embedding_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, embedding_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.net(x), p=2, dim=-1)


class FaceEmbeddingNet(nn.Module):
    """Single nn.Module combining the frozen backbone and trainable head."""

    def __init__(self, backbone: nn.Module, head: EmbeddingHead):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x) # (B, feature_dim)
        embeddings = self.head(features) # (B, embedding_dim), L2-normalised
        return embeddings


class TripletLossWithMetadata(nn.Module):
    """Triplet loss wrapper that logs anchor-level triplet metadata in WeightsLAB.

    The anchor UID is used as batch ID. For each anchor we persist the selected
    positive and negative UIDs into per-sample signal columns so they remain
    attached to image/sample metadata in the ledger.
    """

    def __init__(self, margin: float = 0.3):
        super().__init__()
        self.margin = margin
        self.criterion = nn.TripletMarginLoss(margin=margin, p=2, reduce=False)

    def forward(
        self,
        anchor_emb: torch.Tensor,
        pos_emb: torch.Tensor,
        neg_emb: torch.Tensor,
        anchor_ids: List[str],
        pos_ids: List[str],
        neg_ids: List[str],
        name: str = "train",
    ) -> torch.Tensor:
        loss = self.criterion(anchor_emb, pos_emb, neg_emb)

        wl.save_signals(
            batch_ids=anchor_ids,
            signals={f"{name}/loss/triplet": loss.detach().cpu().numpy()},
            preds_raw=anchor_emb,
            targets=neg_emb,
            preds=None,
            log=True,
        )

        # Store pivot triplet composition as per-sample metadata-like signals.
        wl.save_signals(
            batch_ids=anchor_ids,
            signals={
                f"{name}/meta/triplet_pos_uid": pos_ids,
                f"{name}/meta/triplet_neg_uid": neg_ids,
            },
            preds_raw=None,
            targets=None,
            preds=None,
            log=False,
        )
        return loss


# ============================================================
# High-level model wrapper
# ============================================================

class FaceEmbeddingModel:
    """Wrapper that manages the backbone + head, optimiser, and WeightsLAB tracking.

    Args:
        backbone_name: "resnet18" | "resnet50" | "mobilenet_v3_small"
        embedding_dim: Output embedding dimensionality (default 128).
        head_hidden_dim: Hidden size of the projection MLP (default 256).
        lr: Learning rate for AdamW (default 1e-3).
        weight_decay: AdamW weight decay (default 1e-4).
        freeze_backbone: When True, only the head's parameters receive
                         gradients ? recommended for quick toy runs.
        device: "cpu", "cuda", or "cuda:N".
        pretrained: Load ImageNet-pretrained weights for the backbone.
        margin: Triplet margin (default 0.3).
    """

    def __init__(
        self,
        backbone_name: str = "resnet18",
        embedding_dim: int = 128,
        head_hidden_dim: int = 256,
        lr: float = 1e-3,
        weight_decay: float = 1e-4,
        freeze_backbone: bool = True,
        device: str = "cpu",
        pretrained: bool = True,
        margin: float = 0.3,
    ):
        self.device = torch.device(device)
        self.margin = margin
        self.embedding_dim = embedding_dim

        # ---- Build backbone ----
        weights = "DEFAULT" if pretrained else None
        backbone_feature_dim = self._build_backbone(backbone_name, weights, freeze_backbone)

        # ---- Build head ----
        head = EmbeddingHead(
            in_dim=backbone_feature_dim,
            hidden_dim=head_hidden_dim,
            embedding_dim=embedding_dim,
        )

        # ---- Compose & move to device ----
        self.net = FaceEmbeddingNet(backbone=self._backbone, head=head).to(self.device)

        # ---- WeightsLAB graph tracking ----
        self.net = wl.watch_or_edit(
            self.net,
            flag="model",
            compute_dependencies=False,
            device=str(self.device),
        )

        # ---- WeightsLAB tracked loss ----
        self.triplet_loss_fn = TripletLossWithMetadata(margin=self.margin)
        self.triplet_loss_fn.__name__ = "tripletcriterion"
        watched_loss = wl.watch_or_edit(
            self.triplet_loss_fn,
            flag="loss",
            signal_name="triplet_with_metadata",
            log=False,
            use_batch_ids_as_x=True,
            use_batch_value_as_y=False,
        )
        if watched_loss is not None and hasattr(watched_loss, "forward"):
            self.triplet_loss_fn = watched_loss

        # ---- Optimiser (head-only when backbone is frozen) ----
        trainable = [p for p in self.net.parameters() if p.requires_grad]
        self.optimizer = torch.optim.AdamW(
            trainable, lr=lr, weight_decay=weight_decay
        )

        n_trainable = sum(p.numel() for p in trainable)
        logger.info(
            f"FaceEmbeddingModel | backbone={backbone_name} pretrained={pretrained} "
            f"frozen={freeze_backbone} | emb_dim={embedding_dim} | "
            f"trainable_params={n_trainable:,}"
        )
        print(
            f" Backbone : {backbone_name} (pretrained={pretrained}, frozen={freeze_backbone})\n"
            f" Emb dim : {embedding_dim}\n"
            f" Head dim : {head_hidden_dim}\n"
            f" Trainable : {n_trainable:,} params\n"
            f" Device : {self.device}"
        )

    def _build_backbone(
        self,
        backbone_name: str,
        weights,
        freeze: bool,
    ) -> int:
        """Instantiate backbone, strip its classifier, optionally freeze.

        Sets self._backbone and returns the feature dimensionality.
        """
        if backbone_name == "resnet18":
            base = models.resnet18(weights=weights)
            feature_dim = 512
            base.fc = nn.Identity()
        elif backbone_name == "resnet50":
            base = models.resnet50(weights=weights)
            feature_dim = 2048
            base.fc = nn.Identity()
        elif backbone_name == "mobilenet_v3_small":
            base = models.mobilenet_v3_small(weights=weights)
            feature_dim = 576
            base.classifier = nn.Identity()
        else:
            raise ValueError(
                f"Unsupported backbone {backbone_name!r}. "
                "Choose from 'resnet18', 'resnet50', 'mobilenet_v3_small'."
            )

        if freeze:
            for param in base.parameters():
                param.requires_grad_(False)
            logger.info("Backbone frozen ? training head only.")

        self._backbone = base
        return feature_dim

    # ----------------------------------------------------------
    # Inference
    # ----------------------------------------------------------

    def get_embeddings(self, images: torch.Tensor) -> torch.Tensor:
        """Forward pass in eval mode; returns L2-normalised embeddings (B, D).

        Args:
            images: (B, C, H, W) float tensor (GPU/CPU ? moved internally)

        Returns:
            (B, D) embedding tensor on CPU
        """
        self.net.eval()
        with torch.no_grad():
            emb = self.net(images.to(self.device))
        return emb.cpu()

    # ----------------------------------------------------------
    # Training step
    # ----------------------------------------------------------

    def train_step(
        self,
        images: torch.Tensor,
        labels: torch.Tensor,
        batch_ids: List[str],
        loss_name: str = "triplet",
    ) -> float:
        """One gradient update using online batch-hard triplet mining.

        Args:
            images: (B, C, H, W) float tensor
            labels: (B,) long tensor of identity ids
            batch_ids: list of sample UIDs for WeightsLAB signal logging
            loss_name: "triplet" (contrastive support planned)

        Returns:
            Scalar loss value (Python float)
        """
        self.net.train()
        self.optimizer.zero_grad()

        images = images.to(self.device)
        labels = labels.to(self.device)

        embeddings = self.net(images) # (B, D)

        # Mine hardest triplets in the batch
        anc_idx, pos_idx, neg_idx = mine_batch_hard(embeddings, labels)

        if anc_idx.numel() == 0:
            logger.warning(
                "No valid triplets found in batch (all samples same class?); "
                "skipping gradient step."
            )
            return 0.0

        anc_emb = embeddings[anc_idx]
        pos_emb = embeddings[pos_idx]
        neg_emb = embeddings[neg_idx]

        triplet_ids = [batch_ids[i] for i in anc_idx.tolist()]
        pos_triplet_ids = [batch_ids[i] for i in pos_idx.tolist()]
        neg_triplet_ids = [batch_ids[i] for i in neg_idx.tolist()]

        if loss_name == "triplet":
            loss_by_sample = self.triplet_loss_fn(
                anchor_emb=anc_emb,
                pos_emb=pos_emb,
                neg_emb=neg_emb,
                anchor_ids=triplet_ids,
                pos_ids=pos_triplet_ids,
                neg_ids=neg_triplet_ids,
                name="train",
            )
        else:
            raise ValueError(
                f"Unsupported loss {loss_name!r}. Use 'triplet'."
            )
        # Aggregate per-sample losses into a single scalar and backprop
        loss = loss_by_sample.mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in self.net.parameters() if p.requires_grad],
            max_norm=1.0,
        )
        self.optimizer.step()

        return float(loss.item())

In [ ]:
# ===== face/signals.py =====
"""
Signals (losses and metrics) for face recognition training.

Follows the WeightsLAB pattern: every helper calls wl.save_signals
immediately after computing the value so the platform captures it.

Classes
-------
TripletLosses differentiable loss functions (return torch.Tensor)
FaceMetrics evaluation metrics and clustering-oriented test signals
"""

import numpy as np
import torch
import torch.nn as nn

import weightslab as wl

from typing import Dict



# ============================================================
# Loss functions
# ============================================================

class TripletLosses:
    """Differentiable loss functions for face embedding training."""

    @staticmethod
    def triplet_loss(
        ids,
        anchor_emb: torch.Tensor,
        pos_emb: torch.Tensor,
        neg_emb: torch.Tensor,
        margin: float = 0.3,
        name: str = "train",
    ) -> torch.Tensor:
        """Standard triplet margin loss on pre-mined (a, p, n) embeddings."""
        loss = nn.TripletMarginLoss(margin=margin, p=2)(anchor_emb, pos_emb, neg_emb)
        wl.save_signals(
            batch_ids=ids,
            signals={f"{name}/loss/triplet": loss.detach().cpu().item()},
            preds_raw=anchor_emb,
            targets=neg_emb,
            preds=None,
            log=True,
        )
        return loss

    @staticmethod
    def contrastive_loss(
        ids,
        emb_a: torch.Tensor,
        emb_b: torch.Tensor,
        same_label: torch.Tensor,
        margin: float = 1.0,
        name: str = "train",
    ) -> torch.Tensor:
        """Contrastive loss (Hadsell et al., 2006)."""
        dist = nn.functional.pairwise_distance(emb_a, emb_b)
        loss = (
            same_label * dist.pow(2)
            + (1.0 - same_label) * (margin - dist).clamp(min=0.0).pow(2)
        ).mean()
        wl.save_signals(
            batch_ids=ids,
            signals={f"{name}/loss/contrastive": loss.detach().cpu().item()},
            preds_raw=emb_a,
            targets=emb_b,
            preds=None,
            log=True,
        )
        return loss


# ============================================================
# Evaluation metrics
# ============================================================

class FaceMetrics:
    """Evaluation metrics for face verification, retrieval, and grouping."""

    @staticmethod
    def verification_accuracy(
        ids,
        embeddings: np.ndarray,
        labels: np.ndarray,
        name: str = "test",
    ) -> Dict[str, float]:
        """Pairwise verification accuracy with optimal threshold search."""
        result = compute_verification_metrics(embeddings, labels)
        wl.save_signals(
            batch_ids=ids,
            signals={
                f"{name}/metric/verification_accuracy": result["verification_accuracy"],
                f"{name}/metric/far": result["far"],
                f"{name}/metric/frr": result["frr"],
                f"{name}/metric/best_threshold": result["best_threshold"],
            },
            preds_raw=None,
            targets=None,
            preds=None,
            log=True,
        )
        return result

    @staticmethod
    def rank1_accuracy(
        ids,
        embeddings: np.ndarray,
        labels: np.ndarray,
        name: str = "test",
    ) -> float:
        """1-NN Rank-1 retrieval accuracy (leave-one-out)."""
        acc = compute_rank1_accuracy(embeddings, labels)
        wl.save_signals(
            batch_ids=ids,
            signals={f"{name}/metric/rank1_accuracy": acc},
            preds_raw=None,
            targets=None,
            preds=None,
            log=True,
        )
        return acc

    @staticmethod
    def similarity_grouping_signals(
        ids,
        embeddings: np.ndarray,
        name: str = "test",
        cluster_eps: float = 0.6,
        cluster_min_samples: int = 2,
    ) -> Dict[str, float]:
        """Save per-sample grouping signals for later clustering/sorting in Studio."""
        grouping = compute_similarity_grouping(
            embeddings=embeddings,
            uids=list(ids),
            cluster_eps=cluster_eps,
            cluster_min_samples=cluster_min_samples,
        )

        wl.save_signals(
            batch_ids=ids,
            signals={
                f"{name}/cluster/id": grouping["cluster_id"],
                f"{name}/cluster/size": grouping["cluster_size"],
                f"{name}/cluster/nn1_distance": grouping["nn1_distance"],
                f"{name}/cluster/nn1_uid": grouping["nn1_uid"],
            },
            preds_raw=None,
            targets=None,
            preds=None,
            log=False,
        )

        summary = {
            "num_clusters": grouping["num_clusters"],
            "noise_ratio": grouping["noise_ratio"],
            "mean_nn1_distance": grouping["mean_nn1_distance"],
        }
        wl.save_signals(
            batch_ids=ids,
            signals={
                f"{name}/cluster/num_clusters": summary["num_clusters"],
                f"{name}/cluster/noise_ratio": summary["noise_ratio"],
                f"{name}/cluster/mean_nn1_distance": summary["mean_nn1_distance"],
            },
            preds_raw=None,
            targets=None,
            preds=None,
            log=True,
        )
        return summary

    @staticmethod
    def compute_all_metrics(
        ids,
        embeddings: np.ndarray,
        labels: np.ndarray,
        name: str = "test",
    ) -> Dict[str, float]:
        """Run all metrics and return a merged dict."""
        verif = FaceMetrics.verification_accuracy(ids, embeddings, labels, name=name)
        rank1 = FaceMetrics.rank1_accuracy(ids, embeddings, labels, name=name)
        grouping = FaceMetrics.similarity_grouping_signals(ids, embeddings, name=name)
        return {**verif, "rank1_accuracy": rank1, **grouping}

## 3. Configuration

Every tunable lives in one `config` dict (the inlined `config.yaml`). Wrapping it with
`flag="hyperparameters"` lets the Studio UI read and live-edit these values while training.

In [ ]:
# ============================================================
#  Face Recognition - Training Config (inlined from config.yaml)
# ============================================================
config = {
    "experiment_name": "face_triplet_training",
    "device": "auto",                            # "auto" -> cuda if available, else cpu

    # Training schedule
    "eval_full_to_train_steps_ratio": 50,        # evaluation frequency (steps); -1 disables
    "experiment_dump_to_train_steps_ratio": 10,  # print running loss every N steps
    "is_training": True,                         # start training immediately or not
    "log_every": 10,                             # running-loss print cadence
    "max_steps": 1500,                           # Colab cap so the training cell terminates;
                                                 # set to None to train open-ended (interrupt to stop)

    # WeightsLAB persistence
    # "root_log_dir": leave unset -> auto temp dir
    "enable_h5_persistence": True,

    # WeightsLAB serving
    "serving_grpc": True,
    "serving_cli": True,

    # --------------------------------------------------------
    # Data
    # --------------------------------------------------------
    "data": {
        # dataset_type options:
        #   "olivetti" - sklearn Olivetti Faces (40 ids, 400 imgs, 64x64 grey).
        #                Works offline, no download needed. Ideal for quick tests.
        #   "lfw"      - LFW People via torchvision (download on first run).
        #   "folder"   - Generic ImageFolder layout: root/{train,test}/{class}/*.jpg
        "dataset_type": "olivetti",
        "root_data_dir": "",           # only required for lfw / folder
        "image_size": 64,              # images are resized to image_size x image_size
        "train_ratio": 0.8,            # olivetti only: fraction used for training
        "min_images_per_class": 2,     # identities with fewer samples are discarded

        "train_loader": {
            "batch_size": 32,
            "shuffle": True,
            "n_workers": 0,            # >0 for multi-process loading (not needed for Olivetti)
        },
        "test_loader": {
            "batch_size": 64,
            "shuffle": False,
            "n_workers": 0,
        },
    },

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------
    "model": {
        # backbone: "resnet18" | "resnet50" | "mobilenet_v3_small"
        "backbone": "resnet18",
        "pretrained": True,            # load ImageNet-pretrained weights
        "freeze_backbone": True,       # train the embedding head only (fast & effective)

        # Embedding head
        "embedding_dim": 128,          # output dim of the L2-normalised vector
        "head_hidden_dim": 256,        # hidden dim of the projection MLP

        # Optimiser (AdamW, trainable params only)
        "lr": 0.001,
        "weight_decay": 0.0001,

        # Triplet loss
        "loss": "triplet",
        "margin": 0.3,                 # triplet margin alpha
    },
}

wl.watch_or_edit(config, flag="hyperparameters", defaults=config, poll_interval=1.0)

## 4. Datasets and tracked loaders

Build the **Olivetti Faces** train/test splits and wrap the loaders with
`wl.watch_or_edit(..., flag="data")` so every sample keeps its identity in WeightsLab.
Olivetti downloads automatically through scikit-learn (cached under `~/scikit_learn_data`).

In [ ]:
# ---- Reproduce main.py's __main__ setup ----
# Resolve device ("auto" -> the runtime device detected above).
if config.get("device", "auto") == "auto":
    config["device"] = str(device)

# Log dir (auto temp dir if unset).
if not config.get("root_log_dir"):
    config["root_log_dir"] = tempfile.mkdtemp()
    print(f"No root_log_dir specified - using temp dir: {config['root_log_dir']}")
os.makedirs(config["root_log_dir"], exist_ok=True)

eval_full_to_train_steps_ratio = int(config.get("eval_full_to_train_steps_ratio", 50))
log_every = int(config.get("log_every", 10))
enable_h5 = config.get("enable_h5_persistence", True)

# ---- Datasets ----
data_cfg = config.get("data", {})
dataset_type = data_cfg.get("dataset_type", "olivetti")
root_data = data_cfg.get("root_data_dir", config.get("root_log_dir", "."))
image_size = int(data_cfg.get("image_size", 64))
min_imgs = int(data_cfg.get("min_images_per_class", 2))
train_ratio = float(data_cfg.get("train_ratio", 0.8))

train_dataset = FaceDataset(
    root=root_data,
    dataset_type=dataset_type,
    split="train",
    image_size=image_size,
    train_ratio=train_ratio,
    min_images_per_class=min_imgs,
)
test_dataset = FaceDataset(
    root=root_data,
    dataset_type=dataset_type,
    split="test",
    image_size=image_size,
    train_ratio=train_ratio,
    min_images_per_class=min_imgs,
)

print(
    f"\nDataset : {dataset_type}"
    f" | train={len(train_dataset)}"
    f" | test={len(test_dataset)}"
    f" | classes={train_dataset.num_classes}"
)

train_cfg = data_cfg.get("train_loader", {})
test_cfg = data_cfg.get("test_loader", {})

# ---- WeightsLab-tracked data loaders ----
train_loader = wl.watch_or_edit(
    train_dataset,
    flag="data",
    loader_name="train_loader",
    batch_size=int(train_cfg.get("batch_size", 32)),
    shuffle=train_cfg.get("shuffle", True),
    is_training=True,
    compute_hash=False,
    num_workers=int(train_cfg.get("n_workers", 0)),
    enable_h5_persistence=enable_h5,
)
test_loader = wl.watch_or_edit(
    test_dataset,
    flag="data",
    loader_name="test_loader",
    batch_size=int(test_cfg.get("batch_size", 64)),
    shuffle=test_cfg.get("shuffle", False),
    is_training=False,
    compute_hash=False,
    num_workers=int(test_cfg.get("n_workers", 0)),
    enable_h5_persistence=enable_h5,
)

## 5. Model

A pretrained ResNet-18 backbone (frozen) plus a small L2-normalised embedding head, trained
with online batch-hard **triplet loss**. Constructing it registers the graph and the loss
with WeightsLab.

In [ ]:
# ---- Model ----
model_cfg = config.get("model", {})
model = FaceEmbeddingModel(
    backbone_name=model_cfg.get("backbone", "resnet18"),
    embedding_dim=int(model_cfg.get("embedding_dim", 128)),
    head_hidden_dim=int(model_cfg.get("head_hidden_dim", 256)),
    lr=float(model_cfg.get("lr", 1e-3)),
    weight_decay=float(model_cfg.get("weight_decay", 1e-4)),
    freeze_backbone=model_cfg.get("freeze_backbone", True),
    pretrained=model_cfg.get("pretrained", True),
    margin=float(model_cfg.get("margin", 0.3)),
    device=config["device"],
)

## 6. Train and evaluate steps

`evaluate(...)` collects embeddings over a loader and computes verification / retrieval /
clustering signals; `train(...)` runs the batch-hard triplet loop. The
`guard_training_context` / `guard_testing_context` blocks tag each phase. The loop is capped
by `max_steps` so it terminates on Colab (pass `max_steps=None` to run open-ended).

In [ ]:
def evaluate(
    model: FaceEmbeddingModel,
    loader: torch.utils.data.DataLoader,
    name: str = "test",
) -> Dict[str, float]:
    """Collect embeddings from loader and compute face recognition metrics."""
    print(f"\n{'=' * 55}")
    print(f"Evaluation [{name}]")
    print(f"{'=' * 55}")

    all_embeddings: List[np.ndarray] = []
    all_labels: List[np.ndarray] = []
    all_uids: List[str] = []

    for images, uids, labels, _metadata in loader:
        emb = model.get_embeddings(images)  # (B, D)
        all_embeddings.append(emb.numpy())
        if isinstance(labels, torch.Tensor):
            all_labels.append(labels.numpy())
        else:
            all_labels.append(np.array(labels))
        all_uids.extend(uids if isinstance(uids, list) else list(uids))

    all_embeddings = np.concatenate(all_embeddings, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    metrics = FaceMetrics.compute_all_metrics(
        ids=all_uids,
        embeddings=all_embeddings,
        labels=all_labels,
        name=name,
    )

    print(f" verification_accuracy : {metrics.get('verification_accuracy', float('nan')):.4f}")
    print(f" rank1_accuracy : {metrics.get('rank1_accuracy', float('nan')):.4f}")
    print(f" FAR : {metrics.get('far', float('nan')):.4f}")
    print(f" FRR : {metrics.get('frr', float('nan')):.4f}")
    print(f" best_threshold : {metrics.get('best_threshold', float('nan')):.4f}")
    if "num_clusters" in metrics:
        print(f" num_clusters : {metrics['num_clusters']:.0f}")
        print(f" noise_ratio : {metrics['noise_ratio']:.4f}")
        print(f" mean_nn1_distance : {metrics['mean_nn1_distance']:.4f}")

    return metrics


def train(
    model: FaceEmbeddingModel,
    train_loader: torch.utils.data.DataLoader,
    test_loader: torch.utils.data.DataLoader,
    eval_full_to_train_steps_ratio: int = 50,
    log_every: int = 10,
    loss_name: str = "triplet",
    max_steps: int = None,
) -> Dict[str, Any]:
    """Training loop.

    Runs until `max_steps` optimizer steps have been taken (Colab-friendly), or
    indefinitely when `max_steps` is None (interrupt the cell to stop). Periodic test
    evaluation runs every eval_full_to_train_steps_ratio steps when test_loader is provided.
    """
    print("\n" + "=" * 60)
    print("Face Recognition Training")
    print(f" Loss : {loss_name}")
    print(f" Eval every : {eval_full_to_train_steps_ratio} steps")
    print(f" Max steps : {'infinite (stop with Ctrl+C)' if max_steps is None else max_steps}")
    print("=" * 60)

    data_iter = iter(train_loader)
    losses: List[float] = []
    eval_history: List[Dict[str, Any]] = []
    step = 0

    try:
        while max_steps is None or step < max_steps:
            step += 1

            with guard_training_context:
                # ---- Fetch next batch (cycle loader) ----
                try:
                    images, batch_ids, labels, _metadata = next(data_iter)
                except StopIteration:
                    data_iter = iter(train_loader)
                    images, batch_ids, labels, _metadata = next(data_iter)

                if not isinstance(batch_ids, list):
                    batch_ids = list(batch_ids)

                if isinstance(labels, torch.Tensor):
                    labels_tensor = labels.long()
                else:
                    labels_tensor = torch.tensor(labels, dtype=torch.long)

                # ---- Gradient step ----
                loss_val = model.train_step(
                    images=images,
                    labels=labels_tensor,
                    batch_ids=batch_ids,
                    loss_name=loss_name,
                )
                losses.append(loss_val)

                if step == 1 or step % max(1, log_every) == 0:
                    window = losses[-log_every:] if len(losses) >= log_every else losses
                    print(
                        f"[train] step {step:>7d} "
                        f"| loss={loss_val:.6f} "
                        f"| running_mean={np.mean(window):.6f}"
                    )

                should_eval = (
                    test_loader is not None
                    and (step % eval_full_to_train_steps_ratio == 0)
                )

            if should_eval:
                with guard_testing_context:
                    print(f"\n[eval@test] step {step}")
                    metrics = evaluate(model=model, loader=test_loader, name="test")
                    eval_history.append({"step": step, "metrics": metrics})

    except KeyboardInterrupt:
        print("\nInterrupted by user. Stopping training loop.")

    summary = {
        "train_steps": step,
        "train_loss_mean": float(np.mean(losses)) if losses else float("nan"),
        "train_loss_last": float(losses[-1]) if losses else float("nan"),
        "num_evals": len(eval_history),
    }

    print("\nTraining summary:")
    for k, v in summary.items():
        print(f" {k}: {v}")

    return summary

## 7. Serve and train

`wl.serve(serving_grpc=True, serving_bore=True)` starts the background gRPC server and a
public `bore.pub` tunnel (the endpoint is printed below). `wl.start_training(...)` flips the
experiment into the training state, then the finite loop runs while streaming per-sample
signals. Leave this cell **running** while you watch it in the UI.

In [ ]:
# Start the gRPC backend (+ public bore tunnel) that Weights Studio connects to.
wl.serve(serving_grpc=True, serving_bore=True)

print("\n" + "=" * 60)
print("STARTING FACE RECOGNITION TRAINING")
print(f" Experiment : {config['experiment_name']}")
print(f" Device : {config['device']}")
print(f" Steps : {config.get('max_steps')} | eval_full_to_train_steps_ratio={eval_full_to_train_steps_ratio}")
print(f" Loss : {config.get('model', {}).get('loss', 'triplet')}")
print(f" Logs : {config['root_log_dir']}")
print("=" * 60)

# Flip the experiment into the training state (short timeout keeps this non-blocking).
wl.start_training(timeout=3)

# Colab-friendly finite loop: config["max_steps"] caps the run so the cell terminates.
# Set config["max_steps"] = None (in the Configuration cell) to train open-ended and
# interrupt the cell to stop.
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    eval_full_to_train_steps_ratio=eval_full_to_train_steps_ratio,
    log_every=log_every,
    loss_name=config.get("model", {}).get("loss", "triplet"),
    max_steps=config.get("max_steps", 1500),
)

# Export signal history + per-sample dataframe (like the classification demo).
wl.write_history()
wl.write_dataframe()
print("\nTraining complete. Logs at:", config["root_log_dir"])

## See it live in Weights Studio

**Colab has no Docker daemon**, so run Studio on your own machine and point it at this notebook's
backend using the `bore.pub:<port>` printed in the Expose section:

```bash
pip install weightslab
weightslab ui launch                       # Terminal 1 - opens http://localhost:5173
weightslab tunnel bore.pub:12345           # Terminal 2 - the host:port from the Expose section
```

Then open **http://localhost:5173**. Prefer local-only? Run this example directly on your machine
(`weightslab start example` selects a bundled example) and launch the UI next to it - no tunnel.

---

<div align="center">
Crafted by <a href="https://github.com/GrayboxTech/weightslab">GrayboxTech</a> - if WeightsLab helps you catch a bad label, drop us a star on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a>.
</div>